#### Importing Required Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import joblib

from sklearn.inspection import permutation_importance

#### Loading the Saved Model and Preprocessor

In [2]:
# Loading the final trained stacking regression model.

final_model = joblib.load(
    '../models/final_stacking_model.pkl'
)

# Loading the fitted preprocessing pipeline.

preprocessor = joblib.load(
    '../models/preprocessor.pkl'
)

#### Load the Feature-Engineered Data

In [3]:
# Loading the feature-engineered dataset used for model development.

df = pd.read_csv(
    '../data/processed/engineered_data.csv'
)

df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,Is_Weekend,Inventory_to_Sales_Ratio,Inventory_Gap,Price_Difference,Price_Difference_Percentage,Promotion_Discount,Previous_Demand,Previous_Units_Sold,Rolling_7_Day_Demand,Rolling_7_Day_Sales
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,...,1,1.911765,93,-13.01,-15.175551,0,NaN,NaN,NaN,NaN
1,2022-01-02,S001,P0001,Electronics,North,93,71,0,65.63,5,...,1,1.309859,22,-8.03,-10.901439,0,115.0,102.0,NaN,NaN
2,2022-01-03,S001,P0001,Electronics,North,274,142,229,68.55,15,...,0,1.929577,132,-12.18,-15.087328,15,84.0,71.0,NaN,NaN
3,2022-01-04,S001,P0001,Electronics,North,132,42,0,61.66,10,...,0,3.142857,90,6.78,12.354227,0,132.0,142.0,NaN,NaN
4,2022-01-05,S001,P0001,Electronics,North,319,129,0,59.56,25,...,0,2.472868,190,2.22,3.871643,25,67.0,42.0,NaN,NaN


In [4]:
print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset Shape: (76000, 31)

Columns:
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount', 'Weather Condition', 'Promotion', 'Competitor Pricing', 'Seasonality', 'Epidemic', 'Demand', 'Year', 'Month', 'Week', 'Day', 'Day_of_Week', 'Is_Weekend', 'Inventory_to_Sales_Ratio', 'Inventory_Gap', 'Price_Difference', 'Price_Difference_Percentage', 'Promotion_Discount', 'Previous_Demand', 'Previous_Units_Sold', 'Rolling_7_Day_Demand', 'Rolling_7_Day_Sales']


#### Separate Features and Target

In [5]:
# Separating the target variable from the input features.

X = df.drop(
    columns=['Demand']
)

y = df['Demand']

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (76000, 30)
Target Shape: (76000,)


In [6]:
# Recreating the same train-test split used during model development.

split_index = 63800

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

print("Training Target Shape:", y_train.shape)
print("Testing Target Shape:", y_test.shape)

Training Shape: (63800, 30)
Testing Shape: (12200, 30)
Training Target Shape: (63800,)
Testing Target Shape: (12200,)


#### Transform the Test Data

In [7]:
# Transforming the test features using the fitted preprocessor.

X_test_processed = preprocessor.transform(
    X_test
)

print("Processed Testing Shape:", X_test_processed.shape)

Processed Testing Shape: (12200, 64)


#### Generating Predictions

In [8]:
# Generating predictions from the final stacking model.

y_pred = final_model.predict(
    X_test_processed
)

print("Number of Predictions:", len(y_pred))

Number of Predictions: 12200


#### Permutation Feature Importance

For each feature, the method:

Takes the test data.
Randomly shuffles one feature.
Makes predictions again.
Checks how much the model's performance gets worse.

If shuffling a feature causes a big performance drop:

That feature was important.

If almost nothing changes:

That feature probably wasn't contributing much to the predictions.

##### Calculate Permutation Importance

In [ ]:
# Calculating permutation importance using the test dataset.
# A small number of repeats is used to keep computation manageable.

permutation_result = permutation_importance(
    final_model,
    X_test_processed,
    y_test,
    scoring='neg_mean_absolute_error',
    n_repeats=3,
    random_state=42,
    n_jobs=-1
)

print("Permutation importance calculated.")

d:\Projects\ML and Data Science\Retail Demand Intelligence\.venv\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
